# Wildfire Analysis — Master Notebook

## [ENG] Overview

This is the main notebook for the Wildfire Analysis & ML Pipeline project.
It provides a complete, end-to-end walkthrough of the entire workflow:

1. **Data ingestion** — loading FIRMS fire detections, CLCPlus land cover, Open-Meteo weather
2. **Sensor merging** — combining VIIRS and MODIS into a unified dataset
3. **Confidence mapping** — reconciling VIIRS numerical and MODIS categorical confidence
4. **CLC enrichment** — adding land cover classification to each fire detection
5. **Weather enrichment** — adding historical weather data to each fire detection
6. **Analysis & visualization** — exploring the enriched dataset
7. **Modeling** — training and evaluating the ML model

## [ESP] Descripción general

Este es el notebook principal del proyecto Wildfire Analysis & ML Pipeline.
Proporciona un recorrido completo, de principio a fin, de todo el flujo de trabajo:

1. **Ingesta de datos** — carga de detecciones de fuego FIRMS, cobertura del suelo CLCPlus, clima Open-Meteo
2. **Fusión de sensores** — combinación de VIIRS y MODIS en un dataset unificado
3. **Mapeo de confianza** — reconciliación de confianza numérica VIIRS y categórica MODIS
4. **Enriquecimiento CLC** — adición de clasificación de cobertura del suelo a cada detección
5. **Enriquecimiento climático** — adición de datos de clima histórico a cada detección
6. **Análisis y visualización** — exploración del dataset enriquecido
7. **Modelado** — entrenamiento y evaluación del modelo ML

## [ENG] Project structure / [ESP] Estructura del proyecto

```text
data/raw/firms/          → Raw FIRMS CSVs (Spain/2023, Spain/2024)
data/raw/clcplus/        → CLCPlus GeoTIFF tiles (Spain/2023-2025)
data/processed/merged/   → VIIRS+MODIS merged output
data/processed/enriched/ → Enriched datasets (CLC + weather)
data/processed/predictions/ → Model predictions
src/wildfire/            → Python package (data loading, enrichment, processing)
configs/                 → Configuration files
```

## [ENG] Supporting notebooks / [ESP] Notebooks de soporte

| Notebook | [ENG] Purpose | [ESP] Propósito |
|---|---|---|
| `01_data_exploration.ipynb` | Explore raw FIRMS, CLCPlus, and weather data | Explorar datos crudos FIRMS, CLCPlus y clima |
| `02_enrichment.ipynb` | Walk through the enrichment process step by step | Recorrer el proceso de enriquecimiento paso a paso |
| `03_analysis.ipynb` | Statistical analysis and feature engineering | Análisis estadístico e ingeniería de características |
| `04_modeling.ipynb` | ML model training, evaluation, and predictions | Entrenamiento, evaluación y predicciones del modelo ML |

In [ ]:
import pandas as pd

from wildfire.data.firms import load_all_firms
from wildfire.processing.confidence import add_unified_confidence

In [37]:
df = add_unified_confidence(load_all_firms(country="Spain", years=[2023, 2024]))
display(df)
print(df.columns)

df_no_og = df.drop(columns=["confidence_og_num", "confidence_og_cat"])
display(df)

print(df_no_og.isnull().sum())

# test

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,frp,daynight,type,sensor,country,year,confidence_og_num,confidence_og_cat,confidence_num,confidence_cat
0,43.12620,-4.09110,316.00,1.10,1.00,2023-01-01,221,Aqua,MODIS,92,...,20.40,N,0,modis,Spain,2023,92,<NA>,92,h
1,39.20470,-1.76060,304.40,1.00,1.00,2023-01-01,1326,Aqua,MODIS,59,...,6.20,D,0,modis,Spain,2023,59,<NA>,59,n
2,37.99410,-4.32650,303.50,1.10,1.00,2023-01-01,1326,Aqua,MODIS,53,...,7.10,D,0,modis,Spain,2023,53,<NA>,53,n
3,43.54010,-5.82810,307.30,1.00,1.00,2023-01-02,1105,Terra,MODIS,65,...,10.50,D,2,modis,Spain,2023,65,<NA>,65,n
4,42.38850,-2.72820,301.20,3.10,1.70,2023-01-03,1008,Terra,MODIS,45,...,40.00,D,0,modis,Spain,2023,45,<NA>,45,n
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47500,38.82438,-6.21968,327.26,0.43,0.38,2024-12-31,1332,N20,VIIRS,n,...,2.13,D,0,viirs_noaa20,Spain,2024,<NA>,n,50,n
47501,39.35228,-6.20244,327.92,0.44,0.38,2024-12-31,1332,N20,VIIRS,n,...,6.14,D,0,viirs_noaa20,Spain,2024,<NA>,n,50,n
47502,38.21756,-5.97083,326.27,0.44,0.38,2024-12-31,1332,N20,VIIRS,n,...,2.14,D,0,viirs_noaa20,Spain,2024,<NA>,n,50,n
47503,39.24597,-4.21751,337.61,0.52,0.42,2024-12-31,1332,N20,VIIRS,n,...,8.49,D,0,viirs_noaa20,Spain,2024,<NA>,n,50,n


Index(['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date',
       'acq_time', 'satellite', 'instrument', 'confidence', 'version',
       'brightness_ir', 'frp', 'daynight', 'type', 'sensor', 'country', 'year',
       'confidence_og_num', 'confidence_og_cat', 'confidence_num',
       'confidence_cat'],
      dtype='str')


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,frp,daynight,type,sensor,country,year,confidence_og_num,confidence_og_cat,confidence_num,confidence_cat
0,43.12620,-4.09110,316.00,1.10,1.00,2023-01-01,221,Aqua,MODIS,92,...,20.40,N,0,modis,Spain,2023,92,<NA>,92,h
1,39.20470,-1.76060,304.40,1.00,1.00,2023-01-01,1326,Aqua,MODIS,59,...,6.20,D,0,modis,Spain,2023,59,<NA>,59,n
2,37.99410,-4.32650,303.50,1.10,1.00,2023-01-01,1326,Aqua,MODIS,53,...,7.10,D,0,modis,Spain,2023,53,<NA>,53,n
3,43.54010,-5.82810,307.30,1.00,1.00,2023-01-02,1105,Terra,MODIS,65,...,10.50,D,2,modis,Spain,2023,65,<NA>,65,n
4,42.38850,-2.72820,301.20,3.10,1.70,2023-01-03,1008,Terra,MODIS,45,...,40.00,D,0,modis,Spain,2023,45,<NA>,45,n
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47500,38.82438,-6.21968,327.26,0.43,0.38,2024-12-31,1332,N20,VIIRS,n,...,2.13,D,0,viirs_noaa20,Spain,2024,<NA>,n,50,n
47501,39.35228,-6.20244,327.92,0.44,0.38,2024-12-31,1332,N20,VIIRS,n,...,6.14,D,0,viirs_noaa20,Spain,2024,<NA>,n,50,n
47502,38.21756,-5.97083,326.27,0.44,0.38,2024-12-31,1332,N20,VIIRS,n,...,2.14,D,0,viirs_noaa20,Spain,2024,<NA>,n,50,n
47503,39.24597,-4.21751,337.61,0.52,0.42,2024-12-31,1332,N20,VIIRS,n,...,8.49,D,0,viirs_noaa20,Spain,2024,<NA>,n,50,n


latitude          0
longitude         0
brightness        0
scan              0
track             0
acq_date          0
acq_time          0
satellite         0
instrument        0
confidence        0
version           0
brightness_ir     0
frp               0
daynight          0
type              0
sensor            0
country           0
year              0
confidence_num    0
confidence_cat    0
dtype: int64
